In [3]:
import pandas as pd

from datetime import datetime, timedelta
import requests

In [4]:
# datetime format ------> 2025-01-01T00:00:00Z

In [5]:
# start_datetime = 025-01-01T00:00:00Z
# end_datetime = 2025-12-31T23:59:59Z
# start_datetime = f'{datetime(2025, 1, 1)}T00:00:00Z'

In [6]:
url = "https://data.elexon.co.uk/bmrs/api/v1/datasets/FUELINST"

In [7]:
start_date = datetime(2025, 1, 1)
end_date = datetime(2025, 12, 31)

all_data = []

In [8]:
current_start = start_date

while current_start <= end_date:
    current_end = min(current_start + timedelta(days=6), end_date)

    params = {
        "publishDateTimeFrom": current_start.strftime("%Y-%m-%dT00:00:00Z"),
        "publishDateTimeTo": current_end.strftime("%Y-%m-%dT23:59:59Z"),
        "format": "json"
    }

    response = requests.get(url, params=params)
    response.raise_for_status()

    data = response.json()

    if "data" in data:
        all_data.extend(data["data"])

    current_start = current_end + timedelta(days=1)

df = pd.DataFrame(all_data)

In [9]:
df

,dataset,publishTime,startTime,settlementDate,settlementPeriod,fuelType,generation
0,FUELINST,2025-01-07T23:55:00Z,2025-01-07T23:50:00Z,2025-01-07,48,BIOMASS,1794
1,FUELINST,2025-01-07T23:55:00Z,2025-01-07T23:50:00Z,2025-01-07,48,CCGT,5616
2,FUELINST,2025-01-07T23:55:00Z,2025-01-07T23:50:00Z,2025-01-07,48,COAL,0
3,FUELINST,2025-01-07T23:55:00Z,2025-01-07T23:50:00Z,2025-01-07,48,INTELEC,0
4,FUELINST,2025-01-07T23:55:00Z,2025-01-07T23:50:00Z,2025-01-07,48,INTEW,-531
...,...,...,...,...,...,...,...
2102395,FUELINST,2025-12-31T00:00:00Z,2025-12-30T23:55:00Z,2025-12-30,48,OCGT,0
2102396,FUELINST,2025-12-31T00:00:00Z,2025-12-30T23:55:00Z,2025-12-30,48,OIL,0
2102397,FUELINST,2025-12-31T00:00:00Z,2025-12-30T23:55:00Z,2025-12-30,48,OTHER,285
2102398,FUELINST,2025-12-31T00:00:00Z,2025-12-30T23:55:00Z,2025-12-30,48,PS,-7


In [10]:
# df.to_csv("Exelon.csv", index=False)

In [11]:
df["time_of_day"] = (
    pd.to_timedelta((df["settlementPeriod"] - 1) * 30, unit="m")
    .apply(lambda x: (pd.Timestamp("1970-01-01") + x).time())
)

In [12]:
df

,dataset,publishTime,startTime,settlementDate,settlementPeriod,fuelType,generation,time_of_day
0,FUELINST,2025-01-07T23:55:00Z,2025-01-07T23:50:00Z,2025-01-07,48,BIOMASS,1794,23:30:00
1,FUELINST,2025-01-07T23:55:00Z,2025-01-07T23:50:00Z,2025-01-07,48,CCGT,5616,23:30:00
2,FUELINST,2025-01-07T23:55:00Z,2025-01-07T23:50:00Z,2025-01-07,48,COAL,0,23:30:00
3,FUELINST,2025-01-07T23:55:00Z,2025-01-07T23:50:00Z,2025-01-07,48,INTELEC,0,23:30:00
4,FUELINST,2025-01-07T23:55:00Z,2025-01-07T23:50:00Z,2025-01-07,48,INTEW,-531,23:30:00
...,...,...,...,...,...,...,...,...
2102395,FUELINST,2025-12-31T00:00:00Z,2025-12-30T23:55:00Z,2025-12-30,48,OCGT,0,23:30:00
2102396,FUELINST,2025-12-31T00:00:00Z,2025-12-30T23:55:00Z,2025-12-30,48,OIL,0,23:30:00
2102397,FUELINST,2025-12-31T00:00:00Z,2025-12-30T23:55:00Z,2025-12-30,48,OTHER,285,23:30:00
2102398,FUELINST,2025-12-31T00:00:00Z,2025-12-30T23:55:00Z,2025-12-30,48,PS,-7,23:30:00


In [13]:
df = df.set_index("time_of_day", drop=False)
df

,dataset,publishTime,startTime,settlementDate,settlementPeriod,fuelType,generation,time_of_day
time_of_day,,,,,,,,
23:30:00,FUELINST,2025-01-07T23:55:00Z,2025-01-07T23:50:00Z,2025-01-07,48,BIOMASS,1794,23:30:00
23:30:00,FUELINST,2025-01-07T23:55:00Z,2025-01-07T23:50:00Z,2025-01-07,48,CCGT,5616,23:30:00
23:30:00,FUELINST,2025-01-07T23:55:00Z,2025-01-07T23:50:00Z,2025-01-07,48,COAL,0,23:30:00
23:30:00,FUELINST,2025-01-07T23:55:00Z,2025-01-07T23:50:00Z,2025-01-07,48,INTELEC,0,23:30:00
23:30:00,FUELINST,2025-01-07T23:55:00Z,2025-01-07T23:50:00Z,2025-01-07,48,INTEW,-531,23:30:00
...,...,...,...,...,...,...,...,...
23:30:00,FUELINST,2025-12-31T00:00:00Z,2025-12-30T23:55:00Z,2025-12-30,48,OCGT,0,23:30:00
23:30:00,FUELINST,2025-12-31T00:00:00Z,2025-12-30T23:55:00Z,2025-12-30,48,OIL,0,23:30:00
23:30:00,FUELINST,2025-12-31T00:00:00Z,2025-12-30T23:55:00Z,2025-12-30,48,OTHER,285,23:30:00


In [15]:
df[(df['settlementPeriod'] > 1) & (df['settlementDate'] == '2025-01-01')]

,dataset,publishTime,startTime,settlementDate,settlementPeriod,fuelType,generation,time_of_day
time_of_day,,,,,,,,
23:30:00,FUELINST,2025-01-02T00:00:00Z,2025-01-01T23:55:00Z,2025-01-01,48,BIOMASS,2460,23:30:00
23:30:00,FUELINST,2025-01-02T00:00:00Z,2025-01-01T23:55:00Z,2025-01-01,48,CCGT,2485,23:30:00
23:30:00,FUELINST,2025-01-02T00:00:00Z,2025-01-01T23:55:00Z,2025-01-01,48,COAL,0,23:30:00
23:30:00,FUELINST,2025-01-02T00:00:00Z,2025-01-01T23:55:00Z,2025-01-01,48,INTELEC,0,23:30:00
23:30:00,FUELINST,2025-01-02T00:00:00Z,2025-01-01T23:55:00Z,2025-01-01,48,INTEW,-426,23:30:00
...,...,...,...,...,...,...,...,...
00:30:00,FUELINST,2025-01-01T00:35:00Z,2025-01-01T00:30:00Z,2025-01-01,2,OCGT,0,00:30:00
00:30:00,FUELINST,2025-01-01T00:35:00Z,2025-01-01T00:30:00Z,2025-01-01,2,OIL,0,00:30:00
00:30:00,FUELINST,2025-01-01T00:35:00Z,2025-01-01T00:30:00Z,2025-01-01,2,OTHER,290,00:30:00
